# 20장 RNN, LSTM, GRU — 순서를 읽는 모델

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `rnn_models.py` 와 `p01_compare.py` 를 치고 실행. 20.2의 파라미터 손계산 | [ch20_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `MyLSTMCell.forward` 여섯 줄에 각 게이트가 "무엇을 얼마나"인지 주석으로 | [ch20_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `MyLSTMCell` 복원. 그다음 14장 패키지 `networks/` 에 `rnn.py` 로 넣고 `--model lstm` 으로 등록 | [ch20_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "20"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/20")

준비 끝. 작업 폴더: code/20


## `rnn_models.py` — 20장 순환 신경망: 직접 만든 RNN 셀·LSTM 셀과 PyTorch 내장 nn.LSTM / nn.GRU

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 20장 순환 신경망: 직접 만든 RNN 셀·LSTM 셀과 PyTorch 내장 nn.LSTM / nn.GRU
import torch
import torch.nn as nn


class MyRNNCell(nn.Module):
    """가장 단순한 순환 셀: h_t = tanh(W_x x_t + W_h h_{t-1} + b)"""

    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.wx = nn.Linear(in_dim, hid_dim)
        self.wh = nn.Linear(hid_dim, hid_dim, bias=False)

    def forward(self, x_t, h_prev):
        return torch.tanh(self.wx(x_t) + self.wh(h_prev))


class MyLSTMCell(nn.Module):
    """LSTM 셀: 게이트 셋(잊기 f, 입력 i, 출력 o)과 후보 g. 상태는 (h, c) 둘."""

    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.gates = nn.Linear(in_dim + hid_dim, 4 * hid_dim)   # 네 가지를 한 번에 계산
        self.hid_dim = hid_dim

    def forward(self, x_t, state):
        h_prev, c_prev = state
        z = self.gates(torch.cat([x_t, h_prev], dim=1))          # [B, 4H]
        i, f, g, o = z.chunk(4, dim=1)                            # 넷으로 나눔
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)   # 게이트는 0~1
        g = torch.tanh(g)                                          # 후보 값
        c = f * c_prev + i * g                                    # 기억: 잊을 만큼 잊고, 새것을 더한다
        h = o * torch.tanh(c)                                     # 출력: 기억 중 내보낼 만큼
        return h, c


class MyRNNClassifier(nn.Module):
    """임베딩 → 직접 만든 셀을 시간축으로 반복 → 마지막 h 로 분류"""

    def __init__(self, vocab_size, emb_dim, hid_dim, num_classes, cell="rnn"):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.cell = MyRNNCell(emb_dim, hid_dim) if cell == "rnn" else MyLSTMCell(emb_dim, hid_dim)
        self.kind = cell
        self.fc = nn.Linear(hid_dim, num_classes)
        self.hid_dim = hid_dim

    def forward(self, ids):                                       # ids: [B, L]
        x = self.emb(ids)                                         # [B, L, E]
        B, L, _ = x.shape
        h = x.new_zeros(B, self.hid_dim)
        c = x.new_zeros(B, self.hid_dim)
        lengths = (ids != 0).sum(1)                               # 패딩 제외 실제 길이
        last_h = x.new_zeros(B, self.hid_dim)
        for t in range(L):                                        # 시간축으로 한 토큰씩
            if self.kind == "rnn":
                h = self.cell(x[:, t], h)
            else:
                h, c = self.cell(x[:, t], (h, c))
            is_last = (lengths == t + 1).unsqueeze(1)             # 이 토큰이 문장의 마지막이면
            last_h = torch.where(is_last, h, last_h)              # 그때의 h 를 저장
        return self.fc(last_h)


class TorchRNNClassifier(nn.Module):
    """같은 일을 PyTorch 내장 nn.LSTM / nn.GRU 로. 시간축 반복이 C++ 로 되어 훨씬 빠르다."""

    def __init__(self, vocab_size, emb_dim, hid_dim, num_classes, cell="lstm", bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU, "rnn": nn.RNN}[cell]
        self.rnn = rnn_cls(emb_dim, hid_dim, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(hid_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, ids):
        x = self.emb(ids)                                         # [B, L, E]
        lengths = (ids != 0).sum(1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)                                   # h: 마지막 은닉 상태 [layers*dirs, B, H]
        if isinstance(h, tuple):                                  # LSTM 은 (h, c)
            h = h[0]
        h = torch.cat([h[-2], h[-1]], dim=1) if self.rnn.bidirectional else h[-1]
        return self.fc(h)


if __name__ == "__main__":
    torch.manual_seed(0)
    ids = torch.tensor([[5, 3, 9, 0, 0], [7, 2, 4, 8, 1]])       # 배치 2, 길이 5 (첫 문장은 3토큰 + 패딩)
    for name, m in [("MyRNN", MyRNNClassifier(20, 8, 16, 3, "rnn")),
                    ("MyLSTM", MyRNNClassifier(20, 8, 16, 3, "lstm")),
                    ("nn.LSTM", TorchRNNClassifier(20, 8, 16, 3, "lstm")),
                    ("nn.GRU", TorchRNNClassifier(20, 8, 16, 3, "gru")),
                    ("BiLSTM", TorchRNNClassifier(20, 8, 16, 3, "lstm", bidirectional=True))]:
        n = sum(p.numel() for p in m.parameters())
        print(f"{name:<8} 파라미터 {n:>6,}  [2,5] → {tuple(m(ids).shape)}")
    # 셀 하나의 계산: 입력 8, 은닉 16 일 때 LSTM 파라미터 = (8+16)*64 + 64
    print("MyLSTMCell 파라미터:", sum(p.numel() for p in MyLSTMCell(8, 16).parameters()), "= (8+16)*64+64")
```

In [ ]:
!python rnn_models.py

MyRNN    파라미터    611  [2,5] → (2, 3)
MyLSTM   파라미터  1,811  [2,5] → (2, 3)
nn.LSTM  파라미터  1,875  [2,5] → (2, 3)
nn.GRU   파라미터  1,459  [2,5] → (2, 3)
BiLSTM   파라미터  3,587  [2,5] → (2, 3)
MyLSTMCell 파라미터: 1600 = (8+16)*64+64


## `p01_compare.py` — 20.4 같은 데이터(법령 조문 4종 분류)에서 모델만 바꿔 비교: 단어 주머니 vs 직접 만든 RNN/LSTM vs nn.LSTM/GRU/BiLSTM

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 20.4 같은 데이터(법령 조문 4종 분류)에서 모델만 바꿔 비교: 단어 주머니 vs 직접 만든 RNN/LSTM vs nn.LSTM/GRU/BiLSTM
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "19"))
from statutes_dataset import StatuteDataset
from rnn_models import MyRNNClassifier, TorchRNNClassifier

LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
torch.manual_seed(0)
train_set = StatuteDataset(LAWS, "train")
test_set = StatuteDataset(LAWS, "test", vocab=train_set.vocab)
test_loader = DataLoader(test_set, batch_size=256)
V, C = len(train_set.vocab), len(LAWS)


class BagOfWords(nn.Module):
    def __init__(self, vocab_size, dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, ids):
        vec = self.emb(ids)
        mask = (ids != 0).unsqueeze(-1).float()
        return self.fc((vec * mask).sum(1) / mask.sum(1).clamp(min=1))


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


def run(name, model, epochs=8, lr=2e-3):
    torch.manual_seed(0)
    loader = DataLoader(train_set, batch_size=32, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    best = 0.0
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)     # 기울기 폭발 방지 (RNN 의 관례)
            opt.step()
        best = max(best, accuracy(model))
    n = sum(p.numel() for p in model.parameters())
    print(f"{name:<14} 파라미터 {n:>9,}  최고 정확도 {best:.3f}  마지막 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    print(f"학습 {len(train_set)} / 시험 {len(test_set)} / 어휘 {V}")
    run("단어 주머니", BagOfWords(V, 64, C))
    run("MyRNN", MyRNNClassifier(V, 64, 64, C, "rnn"))
    run("MyLSTM", MyRNNClassifier(V, 64, 64, C, "lstm"))
    run("nn.LSTM", TorchRNNClassifier(V, 64, 64, C, "lstm"))
    run("nn.GRU", TorchRNNClassifier(V, 64, 64, C, "gru"))
    run("BiLSTM", TorchRNNClassifier(V, 64, 64, C, "lstm", bidirectional=True))
```

In [ ]:
!python p01_compare.py

학습 2108 / 시험 526 / 어휘 5986
단어 주머니         파라미터   383,364  최고 정확도 0.892  마지막 0.892  (2초)
MyRNN          파라미터   391,620  최고 정확도 0.751  마지막 0.751  (11초)
MyLSTM         파라미터   416,388  최고 정확도 0.823  마지막 0.823  (13초)
nn.LSTM        파라미터   416,644  최고 정확도 0.852  마지막 0.840  (12초)
nn.GRU         파라미터   408,324  최고 정확도 0.838  마지막 0.838  (11초)
BiLSTM         파라미터   450,180  최고 정확도 0.838  마지막 0.816  (21초)


## 다음 단계

- 2단계 「주석 달기」: [ch20_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch20_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch20_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.